# 2. How do spring differences reach the nodes?

In Energy 01, six nodes formed a chain with five springs. We fixed $z_0=1$
and $z_5=3$ and minimized the energy. Here we start from the same uneven
configuration $[1,1.25,1.5,2,2.25,3]$ and work out how individual springs
contribute to the energy gradient at the nodes.

The signed difference of spring $i$ is $r_i=z_{i+1}-z_i$. Each row of
$D\in\mathbb R^{5\times6}$ subtracts the left node and adds the right node.
Give $D$ to `AffineResponse` with its default zero offset. This gives the
response $r(z)=Dz$, whose Jacobian $J_r=D$ is constant.

| Quantity | Where it lives | Length |
| --- | --- | --- |
| Positions $z$, perturbation $v$ | Nodes | 6 |
| Differences $r(z)$, change $J_rv$ | Springs | 5 |
| Weights $w$ | Springs | 5 |
| Pulled-back weights $J_r^\top w$, energy gradient $\nabla E$ | Nodes | 6 |

In [ ]:
import numpy as np
import scipy.sparse as sp

from litepgo.energy import AffineResponse, QuadraticEnergy

n = 6
z = np.array([1.0, 1.25, 1.5, 2.0, 2.25, 3.0])
tolerance = 1e-10

D = sp.diags(
    [-np.ones(n - 1), np.ones(n - 1)], [0, 1], shape=(n - 1, n), format="csc"
)
response = AffineResponse(D)
r = response.eval_value(z)
expected_r = np.array([0.25, 0.25, 0.5, 0.25, 0.75])

assert response.num_inputs == n and response.num_outputs == n - 1
assert np.max(np.abs(r - expected_r)) < tolerance
print("Node positions", z)
print("Spring differences", r)

## JVP · Move one node and read the changes in the springs

Choose the six-entry direction $v=e_2$ to move only node 2 to the right.
Before running the cell, predict which two spring differences change and
the sign of each change. The Jacobian-vector product (JVP)
$\texttt{eval\_jvp}(z,v)$ returns the five-entry vector $J_rv$. Since the
response is affine, it also equals the difference $r(z+v)-r(z)$ exactly.

In [ ]:
v = np.zeros(n)
v[2] = 1.0
jvp = response.eval_jvp(z, v)

assert np.max(np.abs(jvp - D @ v)) < tolerance
assert np.max(np.abs(jvp - (response.eval_value(z + v) - r))) < tolerance
print("Node direction", v)
print("Change in the springs", jvp)

Spring 1 ends at node 2 and gains one unit. Spring 2 starts at node 2 and
loses one unit.

## VJP · Send five spring weights back to the nodes

Give the springs weights $w=[1,2,4,1,3]$ and consider the scalar
$w^\top r(z)$. Spring $i$ contributes $-w_i$ to its left node and $+w_i$
to its right node. A shared node adds the contributions from both adjacent
springs. At node 2 the result is $+w_1-w_2=2-4=-2$.

The vector-Jacobian product (VJP) $\texttt{eval\_vjp}(z,w)$ returns these
weights as a six-entry column vector $J_r^\top w$. Check the same weighted
change two ways. The left dot product is in spring space. The right one
is in node space.

$$
w^\top(J_rv)=(J_r^\top w)^\top v.
$$

In [ ]:
w = np.array([1.0, 2.0, 4.0, 1.0, 3.0])
vjp = response.eval_vjp(z, w)
spring_dot = w @ jvp
node_dot = vjp @ v

assert np.max(np.abs(vjp - D.T @ w)) < tolerance
assert abs(spring_dot - node_dot) < tolerance
print("Spring weights", w)
print("Weights at the nodes", vjp)
print("Dot product in spring space", spring_dot)
print("Dot product in node space", node_dot)

## Use the VJP to recover the energy gradient

The Energy 01 objective is $E(z)=\tfrac12 r(z)^\top r(z)$. The derivative
of $\tfrac12 r^\top r$ with respect to the five spring differences is $r$,
so use $w=r(z)$ in the VJP. The chain rule gives the six-entry gradient

$$
\nabla E(z)=J_r^\top r(z).
$$

Compare this with the gradient from `QuadraticEnergy` and with centered
finite differences of the scalar energy. For each node $i$, use
$(E(z+h e_i)-E(z-h e_i))/(2h)$ with $h=10^{-6}$. We check all six
partials, including those at the two endpoints. The fixed-end solve in
Energy 01 used only the four interior entries.

In [ ]:
H = (D.T @ D).tocsc()
chain = QuadraticEnergy(H)
gradient_vjp = response.eval_vjp(z, r)
gradient_energy = chain.eval_gradient(z)

h = 1e-6
fd_tolerance = 1e-8
gradient_fd = np.empty(n)
for i in range(n):
    shift = np.zeros(n)
    shift[i] = h
    gradient_fd[i] = (chain.eval_value(z + shift) - chain.eval_value(z - shift)) / (2 * h)

energy_error = np.max(np.abs(gradient_vjp - gradient_energy))
fd_error = np.max(np.abs(gradient_vjp - gradient_fd))
assert energy_error < tolerance
assert fd_error < fd_tolerance
print("Gradient from the response", gradient_vjp)
print("Gradient from the energy", gradient_energy)
print("Gradient from finite differences", gradient_fd)
print(f"Maximum energy API error {energy_error:.3e}")
print(f"Maximum finite difference error {fd_error:.3e}")

## Try it

Give spring 3 weight one and every other spring weight zero. Predict which
two nodes receive a nonzero contribution and what their signs are. Run the
cell, then try a different spring index from 0 to 4.

In [ ]:
spring = 3
trial_weights = np.zeros(n - 1)
trial_weights[spring] = 1.0
trial_nodes = response.eval_vjp(z, trial_weights)
expected_nodes = np.zeros(n)
expected_nodes[spring] = -1.0
expected_nodes[spring + 1] = 1.0

assert np.max(np.abs(trial_nodes - expected_nodes)) < tolerance
print("Spring weights", trial_weights)
print("Weights at the nodes", trial_nodes)

<details>
<summary>Check your prediction</summary>

Spring 3 connects nodes 3 and 4. Its VJP is $[0,0,0,-1,1,0]$.
Changing the spring index moves the same two contributions along the chain.

</details>

**Checkpoint** You can follow a direction from six nodes to five springs with
a JVP, pull five weights back to six nodes with a VJP, and explain why choosing
the spring differences as weights gives the energy gradient.